In [2]:
# ProjectTwoDashboard.ipynb (FULL UPDATED CODE)

from jupyter_dash import JupyterDash
JupyterDash.infer_jupyter_proxy_config()

import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import base64
from pathlib import Path

# IMPORTANT: your CRUD file is animal_crud.py
from animal_crud import AnimalShelter

###########################
# Data / Model
###########################
username = "aacuser"
password = "SNHU1234"

db = AnimalShelter(username, password)

df = pd.DataFrame.from_records(db.read({}))
if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)

###########################
# Rescue Breed Lists
###########################
WATER = ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]
MOUNTAIN = ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog",
            "Siberian Husky", "Rottweiler"]
DISASTER = ["Doberman Pinscher", "German Shepherd",
            "Golden Retriever", "Bloodhound", "Rottweiler"]

def build_query(filter_type):
    base = {
        "animal_type": "Dog",
        "age_upon_outcome_in_weeks": {"$lte": 104}
    }

    if filter_type == "Water Rescue":
        q = dict(base)
        q["breed"] = {"$in": WATER}
        return q

    if filter_type == "Mountain or Wilderness Rescue":
        q = dict(base)
        q["breed"] = {"$in": MOUNTAIN}
        return q

    if filter_type == "Disaster or Individual Tracking":
        q = dict(base)
        q["breed"] = {"$in": DISASTER}
        return q

    return {}

###########################
# ✅ LOGO BLOCK (ROBUST FIX)
###########################
print("Current working directory:", Path.cwd())

def find_logo():
    candidates = [
        Path("code_files") / "Grazioso_Salvare_Logo.png",
        Path("code_files") / "Grazioso Salvare Logo.png",
        Path("code_files") / "GraziosoSalvareLogo.png",
        Path("Grazioso_Salvare_Logo.png"),
        Path("Grazioso Salvare Logo.png"),
    ]

    code_files_dir = Path("code_files")
    if code_files_dir.exists():
        candidates += list(code_files_dir.glob("*.png"))
        candidates += list(code_files_dir.glob("*.PNG"))

    for p in candidates:
        if p.exists() and p.is_file():
            return p
    return None

logo_path = find_logo()

if logo_path:
    with open(logo_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    logo = html.Img(
        src=f"data:image/png;base64,{encoded}",
        style={"height": "100px", "marginBottom": "10px"}
    )
    print("✅ Logo loaded from:", logo_path)
else:
    logo = html.Div(
        "❌ Logo file not found. Check code_files/ and the exact filename (.png) and capitalization.",
        style={"color": "red", "fontWeight": "bold"}
    )
    print("❌ Logo not found. Look inside ./code_files and confirm the filename.")

###########################
# App Layout
###########################
app = JupyterDash(__name__)

app.layout = html.Div([

    html.Div([
        logo,
        html.H2("Grazioso Salvare Dashboard"),
        html.P("Unique Identifier: Kashish G. Prajapati")
    ], style={"textAlign": "center"}),

    html.Hr(),

    html.Div([
        html.Label("Filter by Rescue Type:"),
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Reset (All)", "value": "Reset"},
                {"label": "Water Rescue", "value": "Water Rescue"},
                {"label": "Mountain or Wilderness Rescue", "value": "Mountain or Wilderness Rescue"},
                {"label": "Disaster or Individual Tracking", "value": "Disaster or Individual Tracking"}
            ],
            value="Reset",
            inline=True
        )
    ], style={"padding": "10px"}),

    html.Hr(),

    dash_table.DataTable(
        id="datatable-id",
        columns=[{"name": i, "id": i} for i in df.columns],
        data=df.to_dict("records"),

        page_size=10,
        page_action="native",
        sort_action="native",
        filter_action="native",

        row_selectable="single",
        selected_rows=[],

        style_table={"overflowX": "auto"},
        style_cell={
            "textAlign": "left",
            "padding": "6px",
            "minWidth": "120px",
            "maxWidth": "250px",
            "whiteSpace": "normal"
        }
    ),

    html.Hr(),

    html.Div([
        html.Div(id="graph-id", style={"width": "50%", "padding": "10px"}),
        html.Div(id="map-id", style={"width": "50%", "padding": "10px"})
    ], style={"display": "flex"})
])

###########################
# Callbacks
###########################

@app.callback(
    Output("datatable-id", "data"),
    Input("filter-type", "value")
)
def update_table(filter_type):
    query = build_query(filter_type)

    # Reset shows ALL data
    if filter_type == "Reset":
        query = {}

    results = db.read(query)
    dff = pd.DataFrame.from_records(results)

    if dff.empty:
        return []

    if "_id" in dff.columns:
        dff.drop(columns=["_id"], inplace=True)

    return dff.to_dict("records")


@app.callback(
    Output("graph-id", "children"),
    Input("datatable-id", "data")
)
def update_chart(rows):
    if not rows:
        return html.Div("No data")

    dff = pd.DataFrame(rows)

    if "breed" not in dff.columns:
        return html.Div("Column 'breed' not found")

    fig = px.pie(dff, names="breed", title="Breed Distribution (Current Filter)")
    return dcc.Graph(figure=fig)


@app.callback(
    Output("map-id", "children"),
    [Input("datatable-id", "data"),
     Input("datatable-id", "selected_rows")]
)
def update_map(rows, selected_rows):
    if not rows:
        return html.Div("No location data")

    dff = pd.DataFrame(rows)

    row = 0
    if selected_rows and len(selected_rows) > 0:
        row = selected_rows[0]

    if "location_lat" not in dff.columns or "location_long" not in dff.columns:
        return html.Div("Missing location_lat/location_long columns")

    lat = dff.loc[row, "location_lat"]
    lon = dff.loc[row, "location_long"]

    if pd.isna(lat) or pd.isna(lon):
        return html.Div("Selected animal has no lat/long data")

    breed = dff.loc[row, "breed"] if "breed" in dff.columns else "Unknown"
    name = dff.loc[row, "name"] if "name" in dff.columns else "Unknown"

    return dl.Map(
        center=[30.75, -97.48],
        zoom=10,
        style={"width": "100%", "height": "500px"},
        children=[
            dl.TileLayer(),
            dl.Marker(
                position=[lat, lon],
                children=[
                    dl.Tooltip(str(breed)),
                    dl.Popup([
                        html.H4("Animal Name"),
                        html.P(str(name))
                    ])
                ]
            )
        ]
    )

app.run_server(mode="jupyterlab", port=8050, debug=True)


MongoDB connected (no-auth local connection).
Current working directory: /home/codio/workspace/code_files
✅ Logo loaded from: Grazioso Salvare Logo.png
